# Downlaod data

In [25]:
import io
import zipfile
import requests
import frontmatter

In [26]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [27]:
repo_fnd = read_repo_data('katjaweb', 'MLOps-Fake-News-Detection')
dtc_llm_zoomcamp = read_repo_data('DataTalksClub', 'llm-zoomcamp')

print(f"FND documents: {len(repo_fnd)}")
print(f"FND documents: {len(dtc_llm_zoomcamp)}")

FND documents: 1
FND documents: 48


In [28]:
import re
text = repo_fnd[0]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())

In [29]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [37]:
fnd_chunks = []

for doc in repo_fnd:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    fnd_chunks.extend(chunks)

In [31]:
fnd_chunks[1]

{'start': 1000,
 'chunk': 'ndividuals to critically assess the credibility of the information they encounter online.\n\nThe fake news detection web-service is built using Flask, a web application framework for Python, as the backend server. The server processes user input, specifically the text of a news article, and passes it through a machine-learning model to determine whether the news is real or fake. To provide transparency, the model also outputs a probability score, ensuring users understand that predictions are not always absolute. The service is deployed using AWS Elastic Beanstalk, which manages the infrastructure, scaling, and deployment of the application. The web service can also be tested using a user interface built with Flask, running on a development server.\n\nTo enhance accuracy, standard natural language processing (NLP) techniques were applied to preprocess the text, including tokenization and stopword removal. Additionally, new numerical features were created and 

In [40]:
dtc_llm_zoomcamp_chunks = []

for doc in dtc_llm_zoomcamp:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    dtc_llm_zoomcamp_chunks.extend(chunks)

In [42]:
dtc_llm_zoomcamp_chunks[0]

{'start': 0,
 'chunk': '# Module 1: Introduction\n \nIn this module, we will learn what LLM and RAG are and\nimplement a simple RAG pipeline to answer questions about \nthe FAQ Documents from our Zoomcamp courses\n\nWhat we will do: \n\n* Index Zoomcamp FAQ documents\n    * DE Zoomcamp: https://docs.google.com/document/d/19bnYs80DwuUimHM65UV3sylsCn2j1vziPOwzBwQrebw/edit\n    * ML Zoomcamp: https://docs.google.com/document/d/1LpPanc33QJJ6BSsyxVg-pWNMplal84TdZtq10naIhD8/edit\n    * MLOps Zoomcamp: https://docs.google.com/document/d/12TlBfhIiKtyBv8RnsoJR6F72bkPDGEvPOItJIxaEzE0/edit\n* Create a Q&A system for answering questions about these documents \n\n## 1.1 Introduction to LLM and RAG\n\n<a href="https://www.youtube.com/watch?v=Q75JgLEXMsM&list=PL3MmuxUbc_hIB4fSqLy_0AfTjVLpgjV3R">\n  <img src="https://markdown-videos-api.jorgenkh.no/youtube/Q75JgLEXMsM">\n</a>\n\n* LLM\n* RAG\n* RAG architecture\n* Course outcome\n\n\n## 1.2 Preparing the Environment\n\n<a href="https://www.youtube.com

# Add search

## lexical search

In [46]:
from minsearch import Index

index_fnd = Index(
    text_fields=["chunk"],
    keyword_fields=[]
)

index_fnd.fit(fnd_chunks)

In [47]:
index_llm_zoomcamp = Index(
    text_fields=["chunk"],
    keyword_fields=[]
)

index_llm_zoomcamp.fit(dtc_llm_zoomcamp_chunks)

## Sematnic search

In [48]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [54]:
from tqdm.auto import tqdm
import numpy as np

fnd_embeddings = []

for d in tqdm(fnd_chunks):
    text = d['chunk']
    v = embedding_model.encode(text)
    fnd_embeddings.append(v)

fnd_embeddings = np.array(fnd_embeddings)

  0%|          | 0/17 [00:00<?, ?it/s]

In [55]:
from minsearch import VectorSearch

fnd_vindex = VectorSearch()
fnd_vindex.fit(fnd_embeddings, fnd_chunks)

In [52]:
llm_embeddings = []

for d in tqdm(dtc_llm_zoomcamp_chunks):
    text = d['chunk']
    v = embedding_model.encode(text)
    llm_embeddings.append(v)

llm_embeddings = np.array(llm_embeddings)

  0%|          | 0/260 [00:00<?, ?it/s]

In [56]:
llm_vindex = VectorSearch()
llm_vindex.fit(llm_embeddings, dtc_llm_zoomcamp_chunks)

In [57]:
query = 'Can I join the course now?'
q = embedding_model.encode(query)
results = llm_vindex.search(q)

In [59]:
results[0]

{'start': 2000,
 'chunk': '",\n  \'section\': \'General course-related questions\',\n  \'question\': \'Course - Can I still join the course after the start date?\',\n  \'course\': \'data-engineering-zoomcamp\'},\n {\'text\': \'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.\',\n  \'section\': \'General course-related questions\',\n  \'question\': \'Course - Can I follow the course after it finishes?\',\n  \'course\': \'data-engineering-zoomcamp\'},\n {\'text\': "The purpose of this document is to capture frequently asked technical questions\\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours\'\' live.1\\nSubscribe to course public Google Calendar (it works from Desktop only).\\nRe

## Hybrid search

In [67]:
def text_search(query):
    return index_llm_zoomcamp.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return llm_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)
    
    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)
    
    return combined_results

In [68]:
query = 'Can I still enroll ther course?'
hybrid_search(query)

[{'start': 0,
  'chunk': "Question: I just discovered the couse. can i still enrol\n\nContext:\n\nCourse - Can I still join the course after the start date?\nYes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.\n\nEnvironment - Is Python 3.9 still the recommended version to use in 2024?\nYes, for simplicity (of troubleshooting against the recorded videos) and stability. [source]\nBut Python 3.10 and 3.11 should work fine.\n\nHow can we contribute to the course?\nStar the repo! Share it with friends if you find it useful ❣️\nCreate a PR if you see you can improve the text or the structure of the repository.\n\nAre we still using the NYC Trip data for January 2021? Or are we using the 2022 data?\nWe will use the same data, as the project will essentially remain the same as last year’s. The data is available here\n\nDocker-Compose - 